# 08 — Step 6: reduce and cluster the vibe space

**Purpose.** The 41 vibe scores overlap: dark, bleak, melancholic and slow rise together, and so do funny, slapstick and light. Step 6 asks:
1. **PCA:** how many independent vibe directions are really there, and does recommending with fewer, cleaner directions keep (or improve) accuracy?
2. **Clustering (k-means vs HDBSCAN):** do movies fall into natural vibe groups, and can searching only the groups a user likes make recommendations faster without losing accuracy?

**Every technique is judged in the testing zone** (`07_testing_zone.ipynb`): same 5,000 users, same 2018 cutoff, same metrics, and the result is appended to `artifacts/scoreboard.csv`. The reference is `vibe_match / tier 3` (recall@10 0.0290, recall@50 0.0913). A technique stays only if it is **not clearly worse**, meaning the paired 95% interval includes 0 or is above it, and it brings something: fewer dimensions, or less work per user.

Code: `src/viberec/structure.py`; recommenders in `src/viberec/recommenders.py`.

## 1. Load: vibe scores and the testing-zone benchmark

In [ ]:
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from src.viberec import evaluation as ev, recommenders as rc, structure as st

PROCESSED_DIR = Path("..") / "data" / "processed"
ARTIFACTS_DIR = Path("..") / "artifacts"
MODELS_DIR = Path("..") / "models"
SCOREBOARD = ARTIFACTS_DIR / "scoreboard.csv"
VERSION, SEED = "v1", 42


def write_metadata(path, **info):
    meta = {"created_at": datetime.now(timezone.utc).isoformat(), "produced_by": "notebooks/08_reduce_cluster.ipynb", **info}
    path.with_name(path.stem + ".metadata.json").write_text(json.dumps(meta, indent=2, default=str))


vibe_scores = pd.read_parquet(PROCESSED_DIR / "vibe_scores_v2.parquet").set_index("movieId")
VIBES = [c for c in vibe_scores.columns if c != "source"]
z = rc.standardise(vibe_scores[VIBES].astype(float))                    # each vibe: distance from the average movie

train = pd.read_parquet(PROCESSED_DIR / f"benchmark_train_{VERSION}.parquet")
hits = pd.read_parquet(PROCESSED_DIR / f"benchmark_hits_{VERSION}.parquet").groupby("userId")["movieId"].apply(set).to_dict()
user_mean = train.groupby("userId")["rating"].mean()
at_cutoff = pd.read_parquet(PROCESSED_DIR / f"benchmark_movies_at_cutoff_{VERSION}.parquet").set_index("movieId")
popularity_share = (at_cutoff["rating_count"] / at_cutoff["rating_count"].sum()).to_dict()
pool = np.intersect1d(at_cutoff.index[at_cutoff["tier"] == 3], z.index)   # same pool as vibe_match in notebook 07
vibe_vectors = dict(zip(z.index, z.to_numpy()))                            # diversity is always measured on the 41 vibes
print(f"{len(z):,} movies x {len(VIBES)} vibes; {len(hits):,} test users; pool {len(pool):,} movies")


def score(recommend, pool_size):
    return ev.evaluate(recommend, train, hits, vibe_vectors=vibe_vectors, popularity_share=popularity_share,
                       pool_size=pool_size, return_per_user=True)


reference, reference_users = score(rc.make_vibe_match(z, pool, user_mean), len(pool))
print(f"reference vibe_match: recall@10 {reference['recall@10']:.4f}, recall@50 {reference['recall@50']:.4f} (same as the scoreboard)")

## 2. PCA: how many real vibe directions?

**What PCA does.** It finds new directions through the data, each a weighted mix of vibes that move together. The first direction captures the most variation between movies, the second the most of what is left, and so on. Each direction's **explained variance** is how much of the differences between movies it accounts for.

**Which directions are real: parallel analysis.** Each vibe column is shuffled independently, which destroys any real pattern while keeping each vibe's spread. PCA is run on that noise 20 times. A real direction must explain more than the 95th percentile of the same-numbered direction on noise. This avoids a hand-picked rule such as "keep 90% of the variance".

In [ ]:
PCA_PATH = MODELS_DIR / f"pca_vibes_{VERSION}.joblib"
if PCA_PATH.exists():
    print(f"{PCA_PATH} already exists, loading.")
    pca = joblib.load(PCA_PATH)
    noise = np.array(json.loads(PCA_PATH.with_name(PCA_PATH.stem + ".metadata.json").read_text())["parallel_analysis_95th"])
else:
    start = time.time()
    pca = st.fit_pca(z.to_numpy(), seed=SEED)
    noise = st.parallel_analysis(z.to_numpy(), n_iter=20, quantile=95, seed=SEED)
    joblib.dump(pca, PCA_PATH)
    write_metadata(PCA_PATH, input="vibe_scores_v2 (standardised)", movies=len(z), vibes=VIBES,
                   explained_variance_ratio=pca.explained_variance_ratio_.round(4).tolist(),
                   parallel_analysis_95th=noise.tolist(), runtime_seconds=round(time.time() - start, 1))
n_real = int((pca.explained_variance_ > noise).sum())
cum = np.cumsum(pca.explained_variance_ratio_)
print(f"real directions (beat shuffled noise): {n_real} of {len(VIBES)}; together they explain {cum[n_real - 1]:.1%} of the variation")
print("explained by the first 1/2/4/6/10 directions:", {n: f"{cum[n - 1]:.1%}" for n in (1, 2, 4, 6, 10)})
labels = st.component_labels(pca, VIBES, n_real)
print(labels.assign(explained=labels["explained"].map("{:.1%}".format)).to_string(index=False))

## 3. Testing zone: vibe match on the first n directions

The same vibe_match recommender, but each movie is described by its first n PCA directions instead of all 41 vibes. With all 41 directions it is exactly the reference: PCA only rotates the space, and cosine similarity does not change under rotation. That is a built-in sanity check. The smallest n that is **not clearly worse** than the reference (paired interval for recall@50 includes 0 or is positive) is the useful reduction.

In [ ]:
scores_all = pd.DataFrame(pca.transform(z.to_numpy()), index=z.index)
N_GRID = sorted(set([2, 3, 4, 6, 8, n_real, 12, 16, 24, len(VIBES)]))
pca_rows = []
for n in N_GRID:
    res, users = score(rc.make_vibe_match(scores_all.iloc[:, :n], pool, user_mean, standardize=False), len(pool))
    d50, lo50, hi50, rel50 = ev.paired_gain(users, reference_users, "recall@50", seed=SEED)
    d10, lo10, hi10, rel10 = ev.paired_gain(users, reference_users, "recall@10", seed=SEED)
    pca_rows.append({"n_directions": n, "recall@10": res["recall@10"], "recall@50": res["recall@50"],
                     "vs_ref_recall@10": f"{rel10:+.0%} [{lo10:+.4f}, {hi10:+.4f}]",
                     "vs_ref_recall@50": f"{rel50:+.0%} [{lo50:+.4f}, {hi50:+.4f}]",
                     "not_clearly_worse": hi50 >= 0 and hi10 >= 0, "ms_per_user": res["ms_per_user"]})
    ev.append_scoreboard(SCOREBOARD, "6 reduce & cluster", f"vibe_match / tier 3 / PCA {n} directions", res,
                         settings={"cutoff": "2018-01-04", "users": len(hits), "seed": SEED, "pool_size": len(pool),
                                   "pca_directions": n}, note="PCA on standardised vibe_scores_v2")
pca_eval = pd.DataFrame(pca_rows)
pca_eval.to_csv(ARTIFACTS_DIR / f"pca_testing_zone_{VERSION}.csv", index=False)
print(pca_eval.round(4).to_string(index=False))
ok = pca_eval[pca_eval["not_clearly_worse"]]
print(f"\nsmallest number of directions not clearly worse than all 41 vibes: {ok['n_directions'].min()}")

**Results: PCA**

**10 real vibe directions.** Parallel analysis keeps 10 of 41; together they explain 68% of the differences between movies. They have clear meanings:

| # | Share | One end | Other end | Reads as |
|---|---|---|---|---|
| 1 | 17% | dark, atmospheric, demanding | slapstick, happy ending, romantic | **heavy ↔ light** |
| 2 | 11% | heartwarming, melancholic, romantic | violent, revenge, scary | **tender ↔ brutal** |
| 3 | 8% | fast, happy ending, survival | whimsical, steamy, workplace | **action ↔ quirky** |
| 4 | 7% | funny, satire, slapstick | tragedy, period, slow | **comedy ↔ period tragedy** |
| 5 | 6% | teen, school, suburbia | deadpan, visually striking | **coming-of-age** |
| 6–10 | 3–5% each | marriage / city; future / space; workplace / midlife; family / christmas; … | | setting and life-stage themes |

The first direction is Adil's own **Light↔Heavy** label, found by the data without being told.

**But for recommending, every reduction loses accuracy.** With the 10 "real" directions, recall@50 drops 17%; even with 24 of 41 it drops 7%. All are clearly worse, with paired intervals excluding 0. The speed gain is negligible (0.74 vs 0.81 ms per user). The small directions that look like noise for the catalog as a whole carry exactly the personal detail (workplace, midlife, christmas) that separates one user's taste from another's. With all 41 directions the result equals the reference exactly, which confirms the setup.

**Decision: PCA does not earn its place in the recommender.** The 41 vibes stay. PCA is kept for **understanding and display**: the 10 named directions are a compact way to show a movie's or a user's vibe, for the Torch button or a taste map.

## 4. Clustering: are there natural vibe groups?

**Two methods**
- **k-means:** you choose the number of groups k; it places k centres and assigns every movie to the nearest. It is run on unit-length vectors, so "nearest" follows the same cosine similarity the recommender uses.
- **HDBSCAN:** it finds dense regions (many similar movies close together) as groups, and labels movies in sparse areas as noise. It needs no k, but it needs dense, separated regions to exist.

**Judged by**
- **silhouette** (−1 to 1: is a movie closer to its own group than to the next? above ~0.25 is usually read as real structure);
- **noise share** (HDBSCAN);
- a **sanity check** of the groups' contents;
- **the testing zone** (section 5).

HDBSCAN is tried on the 10 real PCA directions and on all 41 vibes, at four group sizes. Results are cached to `artifacts/` because the 41-vibe runs take about 30 seconds each.

In [ ]:
from sklearn.cluster import HDBSCAN

HDB_PATH = ARTIFACTS_DIR / f"hdbscan_grid_{VERSION}.csv"
if HDB_PATH.exists():
    print(f"{HDB_PATH} already exists, loading.")
    hdb = pd.read_csv(HDB_PATH)
else:
    unit41 = z.to_numpy() / np.linalg.norm(z.to_numpy(), axis=1, keepdims=True)
    spaces = {"10 PCA directions": scores_all.iloc[:, :n_real].to_numpy(), "41 vibes": unit41}
    rows = []
    for space, X in spaces.items():
        for mcs, ms in [(20, 5), (50, 10), (100, None), (250, None)]:
            start = time.time()
            lab = HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit_predict(X)
            sizes = np.bincount(lab[lab >= 0]) if (lab >= 0).any() else np.array([0])
            rows.append({"space": space, "min_cluster_size": mcs, "min_samples": ms, "clusters": int(lab.max() + 1),
                         "noise_share": float((lab < 0).mean()), "largest_cluster": int(sizes.max()),
                         "seconds": round(time.time() - start, 1)})
    hdb = pd.DataFrame(rows)
    hdb.to_csv(HDB_PATH, index=False)
    write_metadata(HDB_PATH, movies=len(z), note="HDBSCAN on standardised vibes (unit length) and on the real PCA directions")
print(hdb.round(2).to_string(index=False))

In [ ]:
K_GRID = [8, 16, 32, 64, 128]
KM_PATH = ARTIFACTS_DIR / f"kmeans_labels_{VERSION}.parquet"
CENT_PATH = ARTIFACTS_DIR / f"kmeans_centroids_{VERSION}.npz"
if KM_PATH.exists() and CENT_PATH.exists():
    print("k-means results already exist, loading.")
    km_labels = pd.read_parquet(KM_PATH).set_index("movieId")
    centroids = dict(np.load(CENT_PATH))
    km_quality = pd.read_csv(ARTIFACTS_DIR / f"kmeans_quality_{VERSION}.csv")
else:
    km_labels, centroids, rows = pd.DataFrame(index=z.index), {}, []
    for k in K_GRID:
        start = time.time()
        lab, cen = st.kmeans_cosine(z.to_numpy(), k, seed=SEED)
        km_labels[f"k{k}"] = lab
        centroids[f"k{k}"] = cen
        sizes = np.bincount(lab)
        rows.append({"k": k, "silhouette": st.silhouette_cosine(z.to_numpy(), lab, seed=SEED),
                     "smallest": int(sizes.min()), "largest": int(sizes.max()), "seconds": round(time.time() - start, 1)})
    km_quality = pd.DataFrame(rows)
    km_labels.reset_index().to_parquet(KM_PATH, index=False)
    np.savez(CENT_PATH, **centroids)
    km_quality.to_csv(ARTIFACTS_DIR / f"kmeans_quality_{VERSION}.csv", index=False)
    write_metadata(KM_PATH, k_grid=K_GRID, seed=SEED, note="k-means on unit-length standardised vibe_scores_v2; one column per k")
print(km_quality.round(3).to_string(index=False))

## 5. Testing zone: does searching only nearby clusters save work without losing hits?

For each user, the taste vector picks the **n_probe** cluster centres closest to it, and only movies in those clusters are scored. This is how an IVF index in a vector database (FAISS, pgvector) speeds up search, so it also previews Step 9.

**Measured:** candidates scanned and ms per user (the saving), and recall@10/50 against full search (the cost). A setting is kept only if it is **not clearly worse** on recall (paired intervals include 0 or are positive).

In [ ]:
cl_rows = []
for k in K_GRID:
    cluster_of = km_labels[f"k{k}"]
    for n_probe in (1, 2, 4):
        if n_probe >= k:
            continue
        rec = rc.make_cluster_vibe_match(vibe_scores[VIBES].astype(float), pool, user_mean, cluster_of,
                                         centroids[f"k{k}"], n_probe=n_probe)
        res, users = score(rec, len(pool))
        d10, lo10, hi10, rel10 = ev.paired_gain(users, reference_users, "recall@10", seed=SEED)
        d50, lo50, hi50, rel50 = ev.paired_gain(users, reference_users, "recall@50", seed=SEED)
        cl_rows.append({"k": k, "n_probe": n_probe, "scanned_share": res["candidates_scanned"] / len(pool),
                        "ms_per_user": res["ms_per_user"], "recall@10": res["recall@10"], "recall@50": res["recall@50"],
                        "vs_full_recall@10": f"{rel10:+.0%} [{lo10:+.4f}, {hi10:+.4f}]",
                        "vs_full_recall@50": f"{rel50:+.0%} [{lo50:+.4f}, {hi50:+.4f}]",
                        "not_clearly_worse": hi10 >= 0 and hi50 >= 0})
        ev.append_scoreboard(SCOREBOARD, "6 reduce & cluster", f"vibe_match / tier 3 / k-means {k}, probe {n_probe}", res,
                             settings={"cutoff": "2018-01-04", "users": len(hits), "seed": SEED, "pool_size": len(pool),
                                       "kmeans_k": k, "n_probe": n_probe}, note="search only the nearest clusters")
cluster_eval = pd.DataFrame(cl_rows)
cluster_eval.to_csv(ARTIFACTS_DIR / f"cluster_testing_zone_{VERSION}.csv", index=False)
print(f"full search: {len(pool):,} candidates, {reference['ms_per_user']:.2f} ms/user")
print(cluster_eval.round(3).to_string(index=False))

### Sanity check: what is inside the clusters?

For one setting (k = 16), each cluster's three most distinctive vibes (highest mean standardised score) and its four most-rated movies.

In [ ]:
titles = pd.read_parquet(PROCESSED_DIR / "movie_tiers_v2.parquet").set_index("movieId")[["title", "rating_count"]]
lab16 = km_labels["k16"]
for c in range(16):
    members = lab16.index[lab16 == c]
    profile = z.loc[members].mean().sort_values(ascending=False).index[:3]
    famous = titles.reindex(members).dropna().nlargest(4, "rating_count")["title"].str.slice(0, 28)
    print(f"cluster {c:2d} ({len(members):5,} movies) | {', '.join(profile):45s} | {' / '.join(famous)}")

**Results: clustering**

**HDBSCAN finds no natural groups.** Across 8 settings it either puts 89% of movies into one blob (35,062 movies) or calls 85–97% of movies noise. Unusable, so rejected.

**k-means: groups are weak but meaningful.** Silhouette 0.09–0.16 for every k, low, confirming there are no sharply separated groups. The vibe space is a continuum (consistent with HDBSCAN). Even so, the sanity check shows the k = 16 regions are recognisable vibe neighbourhoods:
- sci-fi survival (The Matrix, Star Wars, Jurassic Park);
- teen/school (Ferris Bueller, Clueless);
- rom-com/marriage (Pretty Woman, Sleepless in Seattle);
- horror (Scream, The Exorcist);
- uplifting underdog (Forrest Gump, Shawshank);
- dark/violent (Pulp Fiction, Fight Club);
- whimsical (Fantasia, Labyrinth);
- melancholic/slow (American Beauty, Amélie);
- …

A few mixed regions also show up, such as a "campy slapstick" group (Congo, Tank Girl).

**Testing zone: searching only nearby clusters does not pay off.**
- Every setting is clearly worse on recall@50 than full search, from −1% (k = 8, probing 4 of 8 clusters, which scans 52% of the pool) to −29% (k = 128, probing 1).
- The time saving is small or negative: full search over 27,752 movies takes 0.79 ms per user, because one matrix multiplication over 41 numbers per movie is already very fast. The cheapest cluster search takes 0.44 ms, and some settings are slower (up to 1.98 ms) because selecting the clusters costs more than it saves.
- By the rule, none earns its place.

**Decisions**
1. **Recommendation search stays full, over all 41 vibes.** No PCA, no cluster-restricted search. At this catalog size, brute force is fast and loses nothing.
2. **Why clustering may matter later:**
   - with heavier per-movie scoring (CF + reranker in Steps 7 and 10);
   - with much larger vectors;
   - or inside the vector database (Step 9), which uses the same idea as an IVF index.

   It would be re-tested there on the same scoreboard.
3. **Kept for understanding:**
   - the 10 PCA directions (named vibe axes);
   - the k = 16 vibe neighbourhoods.

   Possible uses: picking a varied 30-movie onboarding set (PROJECT.md §6), explaining recommendations, and recommending across neighbourhoods to raise the low diversity seen in notebook 07. Each would be tested when built.

**Limitations.**
- k-means is one partitioning method.
- The speed numbers are for this Mac and this implementation.
- Recall is measured on MovieLens users as a proxy for VibeRec users.